# reglscatterpy — interactive WebGL scatterplots for single-cell data

A guided tour. `reglscatterpy` renders millions of points in the browser and
works on **AnnData / MuData / SpatialData / DataFrame / ndarray**, in Jupyter,
JupyterLab, VS Code and Colab. It drives the *same* widget as the R package
**reglScatterplotR**, so plots look identical across languages.

```bash
pip install reglscatterpy anndata   # numpy, pandas, anywidget come with it
```

Run the cells top to bottom. Each `scatterplot(...)` displays an interactive plot.

## Setup — a small synthetic single-cell dataset

We build one realistic `AnnData` (6 clusters, marker genes, QC metrics) and use
it throughout, so the examples read like a real analysis.

In [38]:
import numpy as np, pandas as pd, anndata as ad
import reglscatterpy as rs

rng = np.random.default_rng(0)
clusters = ['CD4 T','CD8 T','B','NK','Monocyte','Dendritic']
n_per = 500; n = n_per*len(clusters); g = 60
labels = np.repeat(clusters, n_per)
# UMAP: each cluster a Gaussian blob at its own centre
centres = rng.uniform(-8, 8, (len(clusters), 2))
umap = np.repeat(centres, n_per, axis=0) + rng.normal(0, 1.1, (n, 2))
# expression: 10 marker genes per cluster, elevated in that cluster
X = rng.poisson(1.0, (n, g)).astype('float32')
for ci in range(len(clusters)):
    rows = slice(ci*n_per, (ci+1)*n_per); cols = slice(ci*10, ci*10+10)
    X[rows, cols] += rng.poisson(6.0, (n_per, 10))
obs = pd.DataFrame({
    'cluster': pd.Categorical(labels),
    'n_counts': X.sum(1),
    'n_genes': (X>0).sum(1),
    'pct_mito': rng.beta(1.5, 30, n)*100,
}, index=[f'cell{i}' for i in range(n)])
adata = ad.AnnData(X=X, obs=obs, var=pd.DataFrame(index=[f'Gene{i}' for i in range(g)]))
adata.obsm['X_umap'] = umap
adata.obsm['X_pca']  = umap[:, ::-1] + rng.normal(0, 1.5, (n, 2))   # a 2nd embedding
adata

AnnData object with n_obs × n_vars = 3000 × 60
    obs: 'cluster', 'n_counts', 'n_genes', 'pct_mito'
    obsm: 'X_umap', 'X_pca'

## 1. Basics

Colour a UMAP by an `obs` column, or by a gene (read from `.X`).

In [ ]:
rs.scatterplot(adata, x='X_umap', color_by='cluster')

In [ ]:
rs.scatterplot(adata, x='X_umap', color_by='Gene0')   # a marker of CD4 T

## 2. Colour & palettes

Continuous palettes are viridis-family; categorical use ColorBrewer (identical
to the R package). Clip the colour scale with `vmin`/`vmax`.

In [ ]:
rs.scatterplot(adata, x='X_umap', color_by='pct_mito',
    continuous_palette='magma', vmin=0, vmax='p99', legend_title='% mito')

In [ ]:
pal = {cl: h for cl, h in zip(clusters,
    ['#e6194B','#3cb44b','#4363d8','#f58231','#911eb4','#42d4f4'])}
rs.scatterplot(adata, x='X_umap', color_by='cluster', custom_colors=pal)

## 3. Encode size & opacity by data

Map a numeric variable to point **size** or **opacity** on top of colour.

In [ ]:
rs.scatterplot(adata, x='X_umap', color_by='cluster', size_by='n_counts')

In [ ]:
rs.scatterplot(adata, x='X_umap', color_by='cluster', opacity_by='n_genes')

## 4. Styling & legend

Theme the canvas; the legend is a frosted card (tune `legend_opacity` /
`legend_blur`). Its title defaults to the colour variable's name.

In [ ]:
rs.scatterplot(adata, x='X_umap', color_by='cluster',
    title='PBMC — UMAP', background_color='#0d1117', axis_color='#cccccc',
    legend_bg='#1c2128', legend_text='#e6edf3', legend_opacity=0.6,
    legend_blur=12, legend_position='bottom-left')

## 5. Toolbar & zoom-on-selection

`toolbar='left'` adds pan / lasso / zoom-to-selection / reset / screenshot
(drag the grip to move it). `zoom_on_selection=True` auto-frames a lasso.

In [ ]:
rs.scatterplot(adata, x='X_umap', color_by='cluster',
    toolbar='left', zoom_on_selection=True)

## 6. Selection round-trip

Lasso points in the plot, then read them back with `w.selection` — they're
positional indices, so `adata[w.selection]` just works. You can also set the
selection from Python.

In [40]:
w = rs.scatterplot(adata, x='X_umap', color_by='cluster', toolbar='left')
w   # lasso a cluster in the widget

<reglscatterpy._widget._make_class.<locals>.ReglScatter object at 0x7ff14570f690>

In [41]:
# run AFTER lassoing (sample selection so it also works headless):
if not w.selection:
    w.selection = list(np.where(adata.obs['cluster']=='NK')[0][:200])
print(len(w.selection), 'cells selected')
adata[w.selection]            # subset the AnnData (or w.subset())

200 cells selected


View of AnnData object with n_obs × n_vars = 200 × 60
    obs: 'cluster', 'n_counts', 'n_genes', 'pct_mito'
    obsm: 'X_umap', 'X_pca'

## 7. Single-cell workflows on a selection

The select → analyse → annotate loop: lasso a population, then label it, see
its composition, or find its markers — all against the original `adata`.

In [42]:
# composition of the selected cells by cluster
w.composition('cluster')

,count,fraction
cluster,,
NK,200,1.0
B,0,0.0
CD4 T,0,0.0
CD8 T,0,0.0
Dendritic,0,0.0
Monocyte,0,0.0


In [43]:
# top differential genes: selection vs the rest
w.diff_expression(n=8)

,gene,logFC,stat,pval,mean_A,mean_B
0,Gene32,2.145552,28.821191,1.171026e-76,7.190,1.625000
1,Gene39,2.082015,28.743852,8.269239e-77,6.915,1.633214
2,Gene36,2.048435,28.667518,1.327489e-76,6.880,1.663214
3,Gene33,2.115043,28.512662,1.110019e-75,7.120,1.643571
4,Gene31,2.123772,28.388980,1.808711e-75,7.185,1.648571
5,Gene34,2.099226,28.306835,2.349162e-75,7.125,1.662857
6,Gene38,2.039611,26.731925,3.202552e-71,6.910,1.680714
7,Gene30,2.054012,26.682904,7.147134e-71,7.095,1.708571


In [44]:
# write a label back into adata.obs for the selected cells
w.annotate('manual', 'my population')
adata.obs['manual'].value_counts(dropna=False)

manual
NaN              2800
my population     200
Name: count, dtype: int64

## 8. Filtering (distribution sliders)

`filter_by` adds a panel of density sliders — drag to keep cells within ranges
(live), the QC-gating idiom.

In [45]:
rs.scatterplot(adata, x='X_umap', color_by='cluster',
    filter_by=adata.obs[['n_counts','n_genes','pct_mito']])

<reglscatterpy._widget._make_class.<locals>.ReglScatter object at 0x7ff12eb97990>

## 9. Richer tooltips

Show extra fields on hover with `tooltip_by` (obs columns or genes).

In [46]:
rs.scatterplot(adata, x='X_umap', color_by='cluster',
    tooltip_by=['n_counts','pct_mito','Gene0'])

<reglscatterpy._widget._make_class.<locals>.ReglScatter object at 0x7ff12d3fa6d0>

## 10. Linked views (`compose`)

Compare two embeddings side by side — pan/zoom, selection and filtering stay in
sync across the group.

In [47]:
from reglscatterpy import compose
a = rs.scatterplot(adata, x='X_umap', color_by='cluster')
b = rs.scatterplot(adata, x='X_pca',  color_by='cluster')
compose([a, b])

GridBox(children=(<reglscatterpy._widget._make_class.<locals>.ReglScatter object at 0x7ff12eb9e790>, <reglscat…

## 11. Other inputs

DataFrame and ndarray work too; MuData / SpatialData are supported (address a
feature as `'modality:feature'`, an embedding as `'modality:embedding'`).

In [ ]:
df = pd.DataFrame({'x': umap[:,0], 'y': umap[:,1], 'cluster': labels})
rs.scatterplot(df, x='x', y='y', color_by='cluster')

In [ ]:
rs.scatterplot(umap, color_by=labels)        # a numpy array (first 2 cols)

## 12. Export & backends

`enable_download=True` adds a PNG/SVG/PDF button (also on the toolbar). A
`backend='jscatter'` option exists (jupyter-scatter) but the default native
widget is recommended.

In [ ]:
rs.scatterplot(adata, x='X_umap', color_by='cluster', enable_download=True)

## 13. Putting it together

A mini workflow: explore → lasso a population → check composition + markers →
annotate → re-plot coloured by the new annotation.

In [48]:
w2 = rs.scatterplot(adata, x='X_umap', color_by='cluster', toolbar='left')
w2

<reglscatterpy._widget._make_class.<locals>.ReglScatter object at 0x7ff12d830a10>

In [49]:
if not w2.selection:
    w2.selection = list(np.where(adata.obs['cluster']=='B')[0][:300])
print('composition:'); display(w2.composition('cluster'))
print('top markers:');  display(w2.diff_expression(n=5))
w2.annotate('refined', 'B cells')
rs.scatterplot(adata, x='X_umap', color_by='refined')

composition:


,count,fraction
cluster,,
B,300,1.0
CD4 T,0,0.0
CD8 T,0,0.0
Dendritic,0,0.0
Monocyte,0,0.0
NK,0,0.0


top markers:


,gene,logFC,stat,pval,mean_A,mean_B
0,Gene28,2.240507,37.170708,1.181931e-122,6.876667,1.455185
1,Gene23,2.315776,36.904645,1.981257e-120,7.020000,1.410000
2,Gene20,2.260112,36.672458,2.660652e-120,6.903333,1.441111
3,Gene29,2.298612,36.162985,2.626995e-118,6.933333,1.409259
4,Gene26,2.304152,36.014543,1.320547e-117,7.110000,1.439630


<reglscatterpy._widget._make_class.<locals>.ReglScatter object at 0x7ff12d43acd0>